# 조합3(11개 변수) - 시간순 3-Fold 교차검증

`build_leakfree_features.ipynb`로 만든 `train_11features.csv`를 갖고, 팀이 원래
하려던 "3-Fold 시간순 튜닝"을 누수 없는 버전으로 재현한다.

## 폴드 구성 방식

`sklearn.model_selection.TimeSeriesSplit(n_splits=3)`을 사용한 **확장창(expanding
window) 방식**이다. 시간순 정렬된 데이터를 4등분한 뒤:

- Fold 1: 1등분으로 학습 -> 2등분으로 검증
- Fold 2: 1~2등분으로 학습 -> 3등분으로 검증
- Fold 3: 1~3등분으로 학습 -> 4등분으로 검증

항상 "학습 구간이 검증 구간보다 과거"인 것을 보장해서, 폴드를 무작위로 섞는
일반 K-Fold와 달리 미래 데이터로 과거를 검증하는 일이 생기지 않는다. (이미
`train_11features.csv`의 파생변수 자체가 각 행 시점 기준으로 인과적으로 계산돼
있기 때문에, 이렇게 시간순으로만 나누면 폴드를 어떻게 자르든 추가 누수가
생기지 않는다.)

## 임계값(threshold) 처리

팀이 원래 하던 방식과 동일하게, Fold 1(가장 이른 구간)의 검증 결과에서
F1을 최대화하는 임계값을 한 번 정하고, 그 값을 Fold 2·3에도 동일하게 적용해서
Precision/Recall/F1을 비교한다. PR-AUC/ROC-AUC는 임계값과 무관하게 폴드별로
따로 계산한다.

## 주의

- 원래 팀이 얻었던 PR-AUC 0.9768 / 임계값 0.928939 같은 수치는 이번 재현에서는
  다르게 나올 수 있다. 특히 amt_zscore_card, prior_normal_median_amt,
  amt_to_prior_median_ratio가 예전에는 미래 정보가 섞인 버전이었을 가능성이
  있었으므로, 지금 나오는 (아마 더 낮아진) 수치가 오히려 더 신뢰할 수 있는
  숫자다. 성능이 떨어졌다고 실패한 게 아니라, 이제서야 정직한 숫자를 보는
  것이다.
- `amt_to_prior_median_ratio`, `prior_normal_median_amt`에는 NaN이 남아있는데
  (그 카드의 첫 거래, 또는 정상거래 이력이 아예 없는 카드), LightGBM은 NaN을
  네이티브로 처리하므로 별도로 채우지 않는다.

## 0. 경로 설정 및 데이터 로드

In [ ]:
#!pip install lightgbm

  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve,
)

CANDIDATE_TRAIN_FEATURE_PATHS = [
    Path("train_11features.csv"),
    Path("revision/train_11features.csv"),
    Path("../train_11features.csv"),
]


def find_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "train_11features.csv 를 찾지 못했습니다. build_leakfree_features.ipynb를 "
        "먼저 실행했는지, CANDIDATE_TRAIN_FEATURE_PATHS 경로가 맞는지 확인해주세요.\n"
        "시도한 경로:\n" + "\n".join(str(p) for p in candidates)
    )


train_path = find_path(CANDIDATE_TRAIN_FEATURE_PATHS)
print(f"불러온 파일: {train_path.resolve()}")

df = pd.read_csv(train_path, parse_dates=["trans_date_trans_time"])
df = df.sort_values("trans_date_trans_time").reset_index(drop=True)

FEATURE_COLUMNS = [
    "category", "amt", "trans_hour", "age",
    "recent_24h_high_amt_count", "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h", "amt_zscore_card", "prior_normal_median_amt",
    "count_30min", "high_speed",
]
TARGET_COLUMN = "is_fraud"

df["category"] = df["category"].astype("category")

print(f"전체 행 수: {len(df):,}")
print(f"전체 사기율: {df[TARGET_COLUMN].mean()*100:.4f}%")
df[FEATURE_COLUMNS + [TARGET_COLUMN]].head()


불러온 파일: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\revision\train_11features.csv
전체 행 수: 1,296,675
전체 사기율: 0.5789%


,category,amt,trans_hour,age,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,prior_normal_median_amt,count_30min,high_speed,is_fraud
0,misc_net,4.97,0,30,0,NaN,4.97,0.0,NaN,0,0,0
1,grocery_pos,107.23,0,40,0,NaN,107.23,0.0,NaN,1,0,0
2,entertainment,220.11,0,56,0,NaN,220.11,0.0,NaN,1,0,0
3,gas_transport,45.00,0,51,0,NaN,45.00,0.0,NaN,1,0,0
4,misc_pos,41.96,0,32,0,NaN,41.96,0.0,NaN,1,0,0


## 1. 모델 파라미터 (팀이 쓰던 LightGBM 설정 그대로 재사용)

In [2]:
MODEL_PARAMS = dict(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

EARLY_STOPPING_ROUNDS = 50
N_SPLITS = 3


## 2. 시간순 3-Fold 학습 및 평가

In [3]:
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN].astype("int8")

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

fold_results = []
fixed_threshold = None

for fold_idx, (train_idx, valid_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    train_period = (
        df.loc[train_idx, "trans_date_trans_time"].min(),
        df.loc[train_idx, "trans_date_trans_time"].max(),
    )
    valid_period = (
        df.loc[valid_idx, "trans_date_trans_time"].min(),
        df.loc[valid_idx, "trans_date_trans_time"].max(),
    )

    neg = int((y_train == 0).sum())
    pos = int((y_train == 1).sum())
    scale_pos_weight = neg / pos if pos > 0 else 1.0

    print(f"\n===== Fold {fold_idx} =====")
    print(f"Train: {len(X_train):,}건 ({train_period[0]} ~ {train_period[1]}), 사기 {pos}건")
    print(f"Valid: {len(X_valid):,}건 ({valid_period[0]} ~ {valid_period[1]}), 사기 {int(y_valid.sum())}건")
    print(f"scale_pos_weight: {scale_pos_weight:.2f}")

    model = lgb.LGBMClassifier(**MODEL_PARAMS, scale_pos_weight=scale_pos_weight)
    model.fit(
        X_train, y_train,
        eval_X=X_valid, eval_y=y_valid,
        eval_metric="average_precision",
        callbacks=[
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, first_metric_only=True, verbose=False),
        ],
    )

    valid_prob = model.predict_proba(X_valid)[:, 1]
    pr_auc = average_precision_score(y_valid, valid_prob)
    roc_auc = roc_auc_score(y_valid, valid_prob) if y_valid.nunique() > 1 else float("nan")

    # Fold 1(가장 이른 구간)에서만 공통 임계값을 한 번 결정해서 이후 폴드에 그대로 적용
    if fixed_threshold is None:
        precisions, recalls, thresholds = precision_recall_curve(y_valid, valid_prob)
        denom = precisions[:-1] + recalls[:-1]
        f1s = np.divide(
            2 * precisions[:-1] * recalls[:-1], denom,
            out=np.zeros_like(denom), where=denom > 0,
        )
        best_idx = int(np.argmax(f1s)) if len(f1s) else 0
        fixed_threshold = float(thresholds[best_idx]) if len(thresholds) else 0.5
        print(f"[Fold 1 기준으로 결정된 공통 임계값]: {fixed_threshold:.6f}")

    valid_pred = (valid_prob >= fixed_threshold).astype("int8")
    precision = precision_score(y_valid, valid_pred, zero_division=0)
    recall = recall_score(y_valid, valid_pred, zero_division=0)
    f1 = f1_score(y_valid, valid_pred, zero_division=0)
    cm = confusion_matrix(y_valid, valid_pred)

    print(f"Best iteration: {model.best_iteration_}")
    print(f"PR-AUC   : {pr_auc:.6f}")
    print(f"ROC-AUC  : {roc_auc:.6f}")
    print(f"Precision: {precision:.6f} (threshold={fixed_threshold:.6f})")
    print(f"Recall   : {recall:.6f}")
    print(f"F1       : {f1:.6f}")
    print("Confusion Matrix:")
    print(cm)

    fold_results.append(dict(
        fold=fold_idx, train_n=len(X_train), valid_n=len(X_valid),
        pr_auc=pr_auc, roc_auc=roc_auc,
        precision=precision, recall=recall, f1=f1,
        threshold=fixed_threshold,
    ))



===== Fold 1 =====
Train: 324,171건 (2019-01-01 00:00:18 ~ 2019-06-03 19:12:33), 사기 2318건
Valid: 324,168건 (2019-06-03 19:12:41 ~ 2019-10-03 07:36:02), 사기 1509건
scale_pos_weight: 138.85
[Fold 1 기준으로 결정된 공통 임계값]: 0.905704
Best iteration: 481
PR-AUC   : 0.954376
ROC-AUC  : 0.998869
Precision: 0.933333 (threshold=0.905704)
Recall   : 0.890656
F1       : 0.911495
Confusion Matrix:
[[322563     96]
 [   165   1344]]

===== Fold 2 =====
Train: 648,339건 (2019-01-01 00:00:18 ~ 2019-10-03 07:36:02), 사기 3827건
Valid: 324,168건 (2019-10-03 07:36:54 ~ 2020-01-28 15:04:36), 사기 1696건
scale_pos_weight: 168.41
Best iteration: 612
PR-AUC   : 0.971344
ROC-AUC  : 0.999285
Precision: 0.946158 (threshold=0.905704)
Recall   : 0.922170
F1       : 0.934010
Confusion Matrix:
[[322383     89]
 [   132   1564]]

===== Fold 3 =====
Train: 972,507건 (2019-01-01 00:00:18 ~ 2020-01-28 15:04:36), 사기 5523건
Valid: 324,168건 (2020-01-28 15:05:52 ~ 2020-06-21 12:13:37), 사기 1983건
scale_pos_weight: 175.08
Best iteration: 456
PR

## 3. 폴드 요약

In [4]:
results_df = pd.DataFrame(fold_results)
display(results_df)

print("\n=== 폴드 평균 (표준편차) ===")
for col in ["pr_auc", "roc_auc", "precision", "recall", "f1"]:
    print(f"{col:10s}: {results_df[col].mean():.6f} (+/- {results_df[col].std():.6f})")


,fold,train_n,valid_n,pr_auc,roc_auc,precision,recall,f1,threshold
0,1,324171,324168,0.954376,0.998869,0.933333,0.890656,0.911495,0.905704
1,2,648339,324168,0.971344,0.999285,0.946158,0.922170,0.934010,0.905704
2,3,972507,324168,0.977164,0.999456,0.935162,0.945537,0.940321,0.905704



=== 폴드 평균 (표준편차) ===
pr_auc    : 0.967628 (+/- 0.011840)
roc_auc   : 0.999203 (+/- 0.000302)
precision : 0.938218 (+/- 0.006937)
recall    : 0.919454 (+/- 0.027541)
f1        : 0.928609 (+/- 0.015153)
